# 01 — Exploratory Data Analysis
### CREDIT CARD FRAUD DETECTION SYSTEM

Loads the ULB/Kaggle credit card dataset and produces the standard EDA figures + statistics. All numbers are computed from the actual dataset — nothing is hard-coded. Figures are written to `results/figures/eda/`.

> Run from the repository root with the backend venv active:
> `. .venv/bin/activate` (or `.venv\Scripts\activate` on Windows), then select this kernel.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path('backend').resolve()))

from app.fraud_detector.data.loading import load_dataset
from app.fraud_detector.data.quality import assess_quality
from app.fraud_detector.data.eda import run_eda

df, result = load_dataset()  # data/raw/creditcard.csv (auto-download mirror if missing)
print(f"Shape: {df.shape} | source: {result.path}")

In [ ]:
report = assess_quality(df)
print("Quality OK:", report.ok)
print("Errors:", report.errors)
print("Stats:")
for k, v in report.stats.items():
    if not isinstance(v, list):
        print(f"  {k}: {v}")
for w in report.warnings:
    print("  !", w)

In [ ]:
summary = run_eda(df)
print("EDA figures written to results/figures/eda/")
print("Fraud rate:", round(summary['fraud_rate'] * 100, 4), "%")
print("Top correlated features:", {k: round(v, 3) for k, v in summary['top_correlated_features'].items()})

## Findings to note
- The dataset is heavily imbalanced (fraud rate is a fraction of a percent) — accuracy alone is meaningless here.
- `Time` is a seconds offset from the first transaction, not a clock time; we treat it as a 24-hour cycle for pattern exploration.
- `Amount` is heavy-tailed; `log1p` compression is used in the feature pipeline.
- Several V-features (V14, V17, V12, V10) carry most of the signal, which matches what SHAP shows after training.